# Dynamic ACDC shimming: the original pipeline in MRzero and PyTorch

This notebook runs the complete method in Python: original input maps → original
EPI waveform → conventional ACDC initialization → MRzero baseline images →
image-based current optimization → reconstruction → field-limit curve → transfer
to other subjects. **No MATLAB process or optimizer is used.** Saved MATLAB files
are optional numerical comparison references, read with SciPy.

The physical model, current/field constraints, masks, current ordering and image
objective follow the supplied implementation. The original rounded reconstruction
is preserved and audited. The solver changes to PyTorch automatic differentiation.
MRzero is run once per B0 map; its voxel magnetization is reused because the sequence
contains one ideal RF pulse and the currents subsequently change only phase.

| Original component | Python implementation |
|---|---|
| Input selection in `run_code2.m` | `data.py` |
| `make_cartesian_epi_sequence.m` | `sequence.py` |
| `initialize_acdc_currents.m` | `optimization.initialize_acdc_currents` |
| `forward_model.m` | `model.MRzeroACDCModel` |
| `obj_fun.m` | whole-image magnitude MSE |
| `epi_acdc_artCor.m` | `optimization.optimize_currents` |
| `plot_lcurve.m` and transfer scripts | final notebook cells |

Modules are kept small enough to read directly beside this notebook. Units are
meters, seconds, Hz, Hz/A and amperes. Currents have shape **[channel, EPI line]**.

## 1. Imports and settings

`GPU_PROFILE` can be `auto`, `v100`, `gtx1660ti`, or `cpu`. `DEVICE` can be `auto`,
`cuda:0`, or `cpu`. Automatic selection detects CUDA; the GTX preset uses smaller
ADC blocks. Both NVIDIA cards use the same mathematics. CUDA-enabled PyTorch
2.6.0 with its CUDA 11.8 runtime supports these cards; setup commands are in the README.

Default settings reproduce the original subject-006 protocol and run 500 solver
iterations for each original field limit. Change `SHIM_MODE` to 1 (constant),
2 (odd/even), or 3 (one state per line). Mode 2 implements the intended two-state
strategy with consistent shapes and constraints.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter
import json, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import torch
from IPython.display import display, Markdown

HERE = Path.cwd()
if not (HERE/'subject006_slice120_inputs.npz').exists():
    HERE = HERE/'MRzero_validation'
ROOT = HERE.parent
sys.path.insert(0,str(HERE))
from data import load_subject_inputs, default_params
from sequence import make_cartesian_epi_sequence, audit_sequence
from model import MRzeroACDCModel, device_settings, magnitude_mse, magnitude_nrmse_percent
from optimization import initialize_acdc_currents, optimize_currents

SUBJECT = '006'
SLICE = 120                       # Original MATLAB 1-based slice number.
GPU_PROFILE = 'auto'
DEVICE = 'auto'
DTYPE = torch.float32             # Fast GPU model; constraints use float64.
ADC_BLOCK_SIZE = None             # None selects the GPU preset; set an integer to override.
PHASE_METHOD = 'auto'             # Benchmark selects dense or ordered-event integration.
OPTIMIZER = 'auto'                # auto, adam, lbfgs, or adam_lbfgs.
SHIM_MODE = 3
FIELD_LIMITS_HZ = [25,50,100,200,300,500,1000]
ITERATIONS = 500
LEARNING_RATE = 0.05
COMPARE_SAVED_MATLAB = True
TRANSFER_SUBJECTS = ['007','008','009']
RUN_BENCHMARK = True

torch.set_num_threads(4)
torch.manual_seed(0)
settings = device_settings(GPU_PROFILE,DEVICE)
device = settings['device']
block_size = ADC_BLOCK_SIZE or settings['adc_block_size']
versions = {name:version(name) for name in ['MRzeroCore','torch','numpy','scipy']}
print('Device:',device,'| hardware:',settings['name'],'| profile:',settings['profile'])
print('PyTorch:',torch.__version__,'| CUDA runtime:',torch.version.cuda,'| ADC block:',block_size)
if device.type=='cpu':
    print('CPU fallback is active; select a CUDA-enabled kernel to use the GPU.')
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = HERE/'results_python'/run_id
OUTPUT.mkdir(parents=True,exist_ok=False)
report = {'versions':versions,'device':str(device),'hardware':settings['name'],
          'profile':settings['profile'],'gpu_tested':device.type=='cuda',
          'adc_block_size':block_size,'subject':SUBJECT,'slice':SLICE,
          'shim_mode':SHIM_MODE,'iterations':ITERATIONS,'output_directory':str(OUTPUT)}
print('Output:',OUTPUT)

## 2. Original inputs and coordinate conventions

The packaged arrays reproduce the original crop and stride. MPRAGE magnitude is
the initial magnetization, as in the supplied model. It is not a quantitative PD
map. The ACDC basis already contains the coil geometry; X/Y preserve the original
scanner coordinates and the -10 mm Y-origin shift. The input provenance records
the common-frame assumption. Changing anatomy or coil orientation requires an
explicit registration, rather than merely transposing an image.

In [ ]:
inputs = load_subject_inputs(SUBJECT,slice_index=SLICE)
params = default_params(inputs)
params['shim_opt'] = SHIM_MODE
params['niters'] = ITERATIONS
channels = inputs['shim_Hz_per_A'].shape[-1]
lines = params['matrix_size']
constraint_mask = inputs['constraint_mask']  # Same sparse mask as the original optimization.
print('Input shape:',inputs['img'].shape,'| channels:',channels,'| EPI lines:',lines)
print('Head / brain / constraint voxels:',*[int(inputs[k].sum()) for k in
      ['head_mask','brain_mask','constraint_mask']])
fig,axes=plt.subplots(1,3,figsize=(12,4),constrained_layout=True)
for ax,key,title in zip(axes,['img','b0_Hz','constraint_mask'],
                       ['Initial MPRAGE magnitude','Baseline B0 [Hz]','Original constraint mask']):
    im=ax.imshow(inputs[key],cmap='gray' if key!='b0_Hz' else 'coolwarm')
    ax.set_title(title); ax.axis('off'); fig.colorbar(im,ax=ax,shrink=.7)
plt.show()

## 3. EPI sequence generated in Python

The waveform translation preserves the original timing, ADC endpoints, line
events and gradient limits. The designer uses its original gamma constant;
the forward model uses `params['gammaHz']`. The legacy half-grid trajectory and
rounded reconstruction are intentionally retained for this method comparison.
The audit counts dropped samples and overwritten grid entries explicitly.

In [ ]:
seq = make_cartesian_epi_sequence(params,display=True)
trajectory = audit_sequence(seq,lines)
report['trajectory']=trajectory
print(json.dumps(trajectory,indent=2))
np.savez_compressed(OUTPUT/'sequence.npz',**seq)
if COMPARE_SAVED_MATLAB and SUBJECT=='006' and SLICE==120:
    baseline_saved=loadmat(ROOT/'sol_BASELINE.mat',simplify_cells=True)
    for key in baseline_saved['seq']:
        assert np.array_equal(seq[key],baseline_saved['seq'][key]),key
    report['sequence_bitwise_equal_to_saved_matlab']=True
    print('All generated sequence fields exactly match the saved waveform.')

## 4. Conventional ACDC initialization, entirely in PyTorch

This is the same regularized bounded least-squares problem as the original
quadratic program: minimize the masked residual `B0 + basis @ I0`. The channel
bound is `0.99 * Imax`, matching the original initializer. Accelerated projected
gradient solves the small quadratic problem on the selected device. The initializer
does not impose the later dynamic field limit.

In [ ]:
initial = initialize_acdc_currents(inputs['b0_Hz'],inputs['shim_Hz_per_A'],
          current_limit_A=.99*params['Imax'],mask=inputs['head_mask'],device=device)
I0=initial['currents_a']
print('Conventional shim RMS:',initial['rms_before_hz'],'->',initial['rms_after_hz'],'Hz')
print('Initializer converged:',initial['converged'],'| iterations:',initial['iterations'],
      '| time:',round(initial['elapsed_s'],3),'s')
assert initial['converged']
report['initializer']={key:value for key,value in initial.items() if not isinstance(value,torch.Tensor)}

## 5. MRzero baseline and reusable forward model

MRzero simulates one ideal 90-degree pulse, the exact positions, static B0,
gradient moments and T2 relaxation. Diffusion, T2-prime and finite-voxel attenuation
are disabled to match the original point-spin model. Its complex magnetization
at every ADC is cached on the selected device.

During optimization, each voxel receives the integrated ACDC phase
`exp(-2j*pi*sum_c(b_c(r)*integral_0^t(I_c(u) du)))` before receive summation.
The phase includes gradient ramps and inter-line transitions. This factorization
is exact under the original single-RF assumptions; later RF pulses or true
intravoxel field gradients require a different propagation model.

In [ ]:
forward_model = MRzeroACDCModel(inputs,params,seq,device=device,dtype=DTYPE,
                               adc_block_size=block_size,phase_method=PHASE_METHOD)
reference_model = MRzeroACDCModel(inputs,params,seq,b0_hz=np.zeros_like(inputs['b0_Hz']),
                  device=device,dtype=DTYPE,adc_block_size=block_size,phase_method=PHASE_METHOD)
zeros=torch.zeros((channels,lines),device=device,dtype=torch.float64)
constant_currents=I0[:,None].expand(channels,lines)
with torch.no_grad():
    img_ref=reference_model(zeros)
    img_2sh=forward_model(zeros)
    img_acdc=forward_model(constant_currents)
del reference_model
baseline_nrmse={name:magnitude_nrmse_percent(image,img_ref) for name,image in
                [('2SH',img_2sh),('conventional_ACDC',img_acdc)]}
report['baseline_nrmse_percent']=baseline_nrmse
print('Baseline magnitude NRMSE (%):',baseline_nrmse)
fig,axes=plt.subplots(1,3,figsize=(10,4),constrained_layout=True)
for ax,image,title in zip(axes,[img_ref,img_2sh,img_acdc],['B0 = 0 reference','2SH','Conventional ACDC']):
    ax.imshow(image.detach().cpu().abs().numpy(),cmap='gray',vmin=0,vmax=img_ref.abs().max().item())
    ax.set_title(title); ax.axis('off')
plt.show()
np.savez_compressed(OUTPUT/'sol_BASELINE.npz',Ishim=constant_currents.cpu().numpy(),
          img_ref=img_ref.cpu().numpy(),img_2sh=img_2sh.cpu().numpy(),img_acdc=img_acdc.cpu().numpy())

## 6. Benchmark phase integration and optimizer choices

The dense formulation and the ordered-event formulation integrate the same phase.
The second exploits consecutive EPI events to avoid a dense ADC-by-event product.
Which is faster depends on the GPU and precision; `auto` measures both locally.

Adam, L-BFGS and Adam followed by L-BFGS all solve the same magnitude-MSE problem.
The benchmark starts each from the same feasible version of I0 and uses 40 iterations.
It measures actual time and objective improvement, not a universal speed ranking.
`auto` chooses the largest observed MSE improvement per second for this run.

In [ ]:
def synchronize():
    if device.type=='cuda': torch.cuda.synchronize(device)

def time_gradient(method,repetitions=5):
    forward_model.phase_method=method
    def evaluate():
        currents=constant_currents.detach().clone().requires_grad_(True)
        torch.autograd.grad(magnitude_mse(forward_model(currents),img_ref),currents)
    evaluate(); synchronize()
    start=perf_counter()
    for _ in range(repetitions): evaluate()
    synchronize()
    return (perf_counter()-start)/repetitions

phase_times={method:time_gradient(method) for method in ['dense','ordered']}
selected_phase=min(phase_times,key=phase_times.get) if PHASE_METHOD=='auto' else PHASE_METHOD
forward_model.phase_method=selected_phase
report['phase_gradient_seconds']=phase_times
report['phase_method']=selected_phase
print('Forward + autograd times (s):',phase_times,'| selected:',selected_phase)

def solver_budget(method,iterations):
    if method=='adam': return dict(adam_steps=iterations,lbfgs_steps=0)
    if method=='lbfgs': return dict(adam_steps=0,lbfgs_steps=iterations)
    if method=='adam_lbfgs':
        adam_steps=int(.85*iterations)
        return dict(adam_steps=adam_steps,lbfgs_steps=iterations-adam_steps)
    raise ValueError('Unknown optimizer.')

def optimize(limit,method,iterations,callback=None):
    return optimize_currents(forward_model,img_ref,I0,inputs['shim_Hz_per_A'],constraint_mask,
           field_limit_Hz=limit,current_limit_A=params['Imax'],shim_mode=SHIM_MODE,
           device=device,learning_rate=LEARNING_RATE,log_every=100,callback=callback,
           **solver_budget(method,iterations))

optimizer_benchmark={}
if RUN_BENCHMARK:
    # Warm up lazy optimizer initialization before timing the comparison.
    optimize_currents(forward_model,img_ref,I0,inputs['shim_Hz_per_A'],constraint_mask,
          field_limit_Hz=1000,current_limit_A=params['Imax'],shim_mode=SHIM_MODE,
          device=device,adam_steps=1,lbfgs_steps=1)
    for method in ['adam','lbfgs','adam_lbfgs']:
        result=optimize(1000,method,40)
        d=result['diagnostics']
        score=(d['initial_mse']-d['final_mse'])/max(d['elapsed_s'],1e-9)
        optimizer_benchmark[method]={**d,'improvement_per_second':score,
          'nrmse_percent':magnitude_nrmse_percent(forward_model(result['currents_a']).detach(),img_ref)}
        print(method,'| time:',round(d['elapsed_s'],3),'s | NRMSE:',round(optimizer_benchmark[method]['nrmse_percent'],3),'%')
selected_optimizer=(max(optimizer_benchmark,key=lambda name:optimizer_benchmark[name]['improvement_per_second'])
                    if OPTIMIZER=='auto' and optimizer_benchmark else 'adam' if OPTIMIZER=='auto' else OPTIMIZER)
report['optimizer_benchmark']=optimizer_benchmark
report['optimizer']=selected_optimizer
print('Selected optimizer:',selected_optimizer)

## 7. Image-based dynamic optimization with the original constraints

The objective is the same whole-image magnitude MSE, without intensity fitting:
`mean((abs(image(I)) - abs(reference))**2)`.
Both original inequalities remain hard constraints:
`abs(I) <= Imax` and `abs(A @ I) <= field_limit` on the original sampled mask.

The optimizer maps each raw state `z` to
`I = z / max(1, max(abs(z))/Imax, max(abs(A @ z))/field_limit)`.
This covers the original feasible set: every feasible vector is unchanged by the map.
It avoids a large block-diagonal matrix and a penalty approximation. The initial
conventional currents are scaled into the feasible set for each field limit.
Each limit starts independently from I0, as in the original pipeline. The objective
is nonconvex, so changing solvers need not reproduce identical currents or minima.

In [ ]:
results={}
summary=[]
for limit in FIELD_LIMITS_HZ:
    print('Optimizing field limit:',limit,'Hz')
    def progress(record):
        print(' ',record['stage'],'step',record['step'],'| best MSE:',round(record['best_mse'],3))
    solution=optimize(limit,selected_optimizer,ITERATIONS,callback=progress)
    currents=solution['currents_a']
    with torch.no_grad(): image=forward_model(currents)
    fields=inputs['shim_Hz_per_A'].reshape(-1,channels)@currents.cpu().numpy()
    row={'field_limit_Hz':limit,'nrmse_percent':magnitude_nrmse_percent(image,img_ref),
         'initial_mse':solution['diagnostics']['initial_mse'],
         'final_mse':solution['diagnostics']['final_mse'],
         'time_s':solution['diagnostics']['elapsed_s'],
         'max_current_A':float(currents.abs().max().item()),
         'sampled_field_Hz':float(np.max(np.abs(fields[constraint_mask.ravel()]))),
         'full_head_field_Hz':float(np.max(np.abs(fields[inputs['head_mask'].ravel()])))}
    assert row['max_current_A']<=params['Imax']+1e-10
    assert row['sampled_field_Hz']<=limit+1e-8
    assert row['final_mse']<=row['initial_mse']+1e-6
    results[limit]={'currents':currents,'image':image,'solution':solution}
    summary.append(row)
    np.savez_compressed(OUTPUT/f'sol_shimOpt{SHIM_MODE}_ACDCFieldLimit{limit}.npz',
          Ishim2=currents.cpu().numpy(),img_acdc2=image.cpu().numpy(),
          shim_fields_Hz=fields.reshape(*inputs['img'].shape,lines))
    (OUTPUT/f'optimization_{limit}Hz.json').write_text(json.dumps(
          {'history':solution['history'],'diagnostics':solution['diagnostics']},indent=2))
    print('Finished:',round(row['nrmse_percent'],3),'% NRMSE |',round(row['time_s'],3),'s')
report['optimization_results']=summary
table='| Field limit Hz | NRMSE % | Time s | max current A | sampled field Hz | full-head field Hz |\n|---:|---:|---:|---:|---:|---:|\n'
for row in summary:
    table+='| {field_limit_Hz} | {nrmse_percent:.2f} | {time_s:.2f} | {max_current_A:.3f} | {sampled_field_Hz:.2f} | {full_head_field_Hz:.2f} |\n'.format(**row)
display(Markdown(table))

## 8. Images, currents, fields and field-limit curve

The image panels share a magnitude scale. The curve uses the same NRMSE definition
as the original plotting script. Saved MATLAB results are a comparison of solver
outcomes; they are not reused as the initial guess or optimized Python result.

In [ ]:
fig,ax=plt.subplots(figsize=(7,4),constrained_layout=True)
ax.plot(FIELD_LIMITS_HZ,[r['nrmse_percent'] for r in summary],'-o',label='Python / MRzero optimization')
ax.axhline(baseline_nrmse['2SH'],ls='--',label='2SH')
ax.axhline(baseline_nrmse['conventional_ACDC'],ls='--',label='Conventional ACDC')
if COMPARE_SAVED_MATLAB and SUBJECT=='006' and SHIM_MODE==3:
    saved_nrmse=[]
    saved_ref=loadmat(ROOT/'sol_BASELINE.mat',variable_names=['img_ref'],simplify_cells=True)['img_ref']
    for limit in FIELD_LIMITS_HZ:
        d=loadmat(ROOT/f'sol_shimOpt3_ACDCFieldLimit{limit}.mat',variable_names=['img_acdc2'],simplify_cells=True)
        saved_nrmse.append(float(100*np.linalg.norm(np.abs(d['img_acdc2'])-np.abs(saved_ref))/np.linalg.norm(saved_ref)))
    ax.plot(FIELD_LIMITS_HZ,saved_nrmse,':s',label='Saved MATLAB optimizer')
    report['saved_matlab_nrmse_percent']=dict(zip(map(str,FIELD_LIMITS_HZ),saved_nrmse))
ax.set(xlabel='Sampled ACDC field limit [Hz]',ylabel='Magnitude NRMSE [%]',title='Original field-limit tradeoff')
ax.legend(); fig.savefig(OUTPUT/'field_limit_curve.png',dpi=160); plt.show()

shown=[FIELD_LIMITS_HZ[0],FIELD_LIMITS_HZ[len(FIELD_LIMITS_HZ)//2],FIELD_LIMITS_HZ[-1]]
images=[img_ref,img_2sh,img_acdc]+[results[limit]['image'] for limit in shown]
titles=['B0 = 0','2SH','Conventional ACDC']+[f'Optimized, {limit} Hz' for limit in shown]
fig,axes=plt.subplots(1,len(images),figsize=(17,4),constrained_layout=True)
for ax,image,title in zip(axes,images,titles):
    ax.imshow(image.detach().cpu().abs().numpy(),cmap='gray',vmin=0,vmax=img_ref.abs().max().item())
    ax.set_title(title,fontsize=10); ax.axis('off')
fig.savefig(OUTPUT/'reconstructed_images.png',dpi=160); plt.show()

selected_limit=FIELD_LIMITS_HZ[-1]
currents=results[selected_limit]['currents'].cpu().numpy()
fields=(inputs['shim_Hz_per_A'].reshape(-1,channels)@currents).reshape(*inputs['img'].shape,lines)
fig,ax=plt.subplots(figsize=(10,4),constrained_layout=True)
ax.plot(np.arange(1,lines+1),currents.T)
ax.set(xlabel='EPI line / ACDC event',ylabel='Current [A]',title=f'Optimized currents, {selected_limit} Hz limit')
plt.show()
fig,axes=plt.subplots(4,8,figsize=(16,8),constrained_layout=True)
for line,ax in enumerate(axes.flat):
    if line<lines:
        im=ax.imshow(fields[...,line],cmap='coolwarm',vmin=-selected_limit,vmax=selected_limit)
        ax.set_title(f'Line {line+1}',fontsize=9)
    ax.axis('off')
fig.colorbar(im,ax=axes.ravel().tolist(),shrink=.6,label='ACDC field [Hz]')
plt.show()

## 9. Original transfer experiment, using the new Python currents

This reproduces the original transfer question: apply subject-006 waveforms to
the other subjects without reoptimizing them. Each target subject has its own
conventional initialization and MRzero baseline. Transferred fields need not
respect the field mask constraints of another subject and can worsen its image.

In [ ]:
transfer=[]
for subject in TRANSFER_SUBJECTS:
    target_inputs=load_subject_inputs(subject,slice_index=SLICE)
    target_params=default_params(target_inputs)
    target_seq=make_cartesian_epi_sequence(target_params)
    target_model=MRzeroACDCModel(target_inputs,target_params,target_seq,device=device,dtype=DTYPE,
                  adc_block_size=block_size,phase_method=selected_phase)
    target_reference=MRzeroACDCModel(target_inputs,target_params,target_seq,
                  b0_hz=np.zeros_like(target_inputs['b0_Hz']),device=device,dtype=DTYPE,
                  adc_block_size=block_size,phase_method=selected_phase)
    target_initial=initialize_acdc_currents(target_inputs['b0_Hz'],target_inputs['shim_Hz_per_A'],
                  .99*target_params['Imax'],target_inputs['head_mask'],device=device)
    with torch.no_grad():
        target_ref=target_reference(zeros)
        target_static=target_model(target_initial['currents_a'][:,None].expand(channels,lines))
        row={'subject':subject,'conventional_nrmse_percent':magnitude_nrmse_percent(target_static,target_ref)}
        for limit in FIELD_LIMITS_HZ:
            image=target_model(results[limit]['currents'])
            row[f'transfer_{limit}Hz_nrmse_percent']=magnitude_nrmse_percent(image,target_ref)
            np.savez_compressed(OUTPUT/f'subject{subject}_transfer_{limit}Hz.npz',img_acdc2=image.cpu().numpy())
    transfer.append(row)
    print(subject,row)
    del target_model,target_reference
report['transfer_results']=transfer

## 10. Saved outputs and interpretation

The run directory contains the waveform, baseline, optimized current arrays,
complex images, field maps, objective histories, timing measurements and transfer
results. Constraints are checked on the original sampled mask, with their actual
full-head peaks reported separately. Changing to a full-head mask, ±3 A, or a
measured current response is a subsequent study, rather than a silent change here.

This reproduces the supplied method with a new GPU solver. It preserves its known
trajectory limitations and does not demonstrate intravoxel signal recovery.
Native MRzero and automatic-gradient checks are recorded in
`python_validation_cpu.json` and, when available, `python_validation_gpu.json`.

Sources: [MRzero voxel magnetization](https://github.com/MRsources/MRzero-Core/blob/main/python/MRzeroCore/simulation/main_pass.py),
[PyTorch complex autograd](https://docs.pytorch.org/docs/stable/notes/autograd.html),
[Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html),
[L-BFGS](https://docs.pytorch.org/docs/stable/generated/torch.optim.LBFGS.html).

In [ ]:
report['parameters']={key:float(value) if isinstance(value,(float,int)) else value
                      for key,value in params.items() if key not in ['X','Y']}
report['elapsed_optimization_s']=sum(row['time_s'] for row in summary)
if device.type=='cuda': report['peak_gpu_memory_gb']=torch.cuda.max_memory_allocated(device)/2**30
(OUTPUT/'report.json').write_text(json.dumps(report,indent=2))
(HERE/'python_pipeline_report.json').write_text(json.dumps(report,indent=2))
print('Report saved:',OUTPUT/'report.json')
print('Total optimization time:',round(report['elapsed_optimization_s'],3),'s')